# Flow

# Problem Stmt

# Complete Architecture

                    decoder_input
                         │
                         ▼
             ┌──────────────────────┐
             │ Decoder Multi-Head   │
             │ Causal Self-Attention│
             └──────────┬───────────┘
                        │
                        ▼
             decoder_input + attention
                        │
                        ▼
                    LayerNorm
                        │
                        ▼
                       x1
                        │
                        ▼
             ┌──────────────────────┐
             │    Cross-Attention   │◄──── encoder_output
             │ Q = x1               │
             │ K,V = encoder_output │
             └──────────┬───────────┘
                        │
                        ▼
                  x1 + attention
                        │
                        ▼
                    LayerNorm
                        │
                        ▼
                       x2
                        │
                        ▼
             ┌──────────────────────┐
             │         FFN          │
             └──────────┬───────────┘
                        │
                        ▼
                   x2 + FFN
                        │
                        ▼
                    LayerNorm
                        │
                        ▼
                  decoder_output

# Function

In [1]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """
    Numerically stable softmax.
    """
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    softmax=exp_x / np.sum(exp_x, axis=axis, keepdims=True)
    return softmax

def causal_mask(seq_len):
    causal_mask = np.triu(
        np.full((seq_len, seq_len), -np.inf),
        k=1
    )
    return causal_mask

def layer_norm(x: np.ndarray, gamma: np.ndarray, beta: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """
    Returns the last-axis normalized array.
    """
    mean=np.mean(x,axis=-1,keepdims=True)
    var=np.var(x,axis=-1,keepdims=True)
    x_norm=(x-mean)/np.sqrt(var+eps)
    layernorm=(gamma*x_norm)+beta
    return layernorm

def relu(x):
    return np.maximum(0, x)

def feed_forward(x: np.ndarray, W1: np.ndarray, b1: np.ndarray,
                 W2: np.ndarray, b2: np.ndarray) -> np.ndarray:
    """
    Returns the position-wise feed-forward output.
    """
    h1=np.add((x@W1),b1)
    h1_relu=relu(h1)
    ffn=np.add((h1_relu@W2),b2)
    return ffn
def residual(x,att_x):
    return x+att_x
    
def decoder_multi_head_attention(
    X,
    wq,
    wk,
    wv,
    wo,
    num_heads
):
    # Create QKV from input X
    Q=X@wq
    K=X@wk
    V=X@wv
    # define shape 
    batch_size, seq_length, d_model = Q.shape
    # define d_head
    d_head=d_model//num_heads
    # creation of heads
    Q_heads = Q.reshape(
        batch_size, seq_length, num_heads, d_head
    )

    K_heads = K.reshape(
        batch_size, seq_length, num_heads, d_head
    )

    V_heads = V.reshape(
        batch_size, seq_length, num_heads, d_head
    )
    # transpose
    Q_heads = Q_heads.transpose(0, 2, 1, 3)
    K_heads = K_heads.transpose(0, 2, 1, 3)
    V_heads = V_heads.transpose(0, 2, 1, 3)

    # q and kt dot product
    scores = Q_heads @ K_heads.transpose(0, 1, 3, 2)

    # Scale by sqrt(d_head)
    scores = scores / np.sqrt(d_head)

    # Causal mask
    causal = causal_mask(seq_length)

    # Apply mask
    masked = scores + causal

    #Soft masx
    attention_weights = softmax(masked, axis=-1)
    # weigthed sum of v
    head_outputs = attention_weights @ V_heads
    # transpose
    head_outputs = head_outputs.transpose(0, 2, 1, 3)
    #concate
    head_outputs = head_outputs.reshape(
        batch_size, seq_length, d_model
    )

    # Output projection

    output_self = head_outputs @ wo

    return output_self

def cross_attention(
    decoder_input,
    encoder_output,
    Wq,
    Wk,
    Wv,
    Wo,
    num_heads
):
    Q=decoder_input @ Wq
    K=encoder_output @ Wk
    V=encoder_output @ Wv

    batch_size, seq_length, d_model = Q.shape
    batch_size_en, seq_length_en, d_model_en = K.shape
    # define d_head
    d_head=d_model//num_heads

    Q_head=Q.reshape(batch_size,seq_length,num_heads,d_head)
    K_head=K.reshape(batch_size_en,seq_length_en,num_heads,d_head)
    V_head=V.reshape(batch_size_en,seq_length_en,num_heads,d_head)

    Q_head=Q_head.transpose(0,2,1,3)
    K_head=K_head.transpose(0,2,1,3)
    V_head=V_head.transpose(0,2,1,3)

    # q and kt dot product
    score=Q_head @ K_head.transpose(0,1,3,2)
    score=score/np.sqrt(d_head)

    # apply softmax
    score_soft=softmax(score,axis=-1)
    attentions=score_soft @ V_head

    head_output=attentions.transpose(0,2,1,3)
    head_output=head_output.reshape(batch_size, seq_length, d_model)
    
    output_cross=head_output@Wo
    return output_cross

def decoder_block(
    decoder_input,
    encoder_output,

    Wq_self,
    Wk_self,
    Wv_self,
    Wo_self,

    Wq_cross,
    Wk_cross,
    Wv_cross,
    Wo_cross,

    W1,
    b1,
    W2,
    b2,

    gamma1,
    beta1,
    gamma2,
    beta2,
    gamma3,
    beta3,

    num_heads
):
    self_attention=decoder_multi_head_attention(decoder_input,Wq_self,Wk_self,Wv_self,Wo_self,num_heads)
    first_residual = residual(decoder_input, self_attention)
    first_layer = layer_norm(first_residual, gamma1, beta1)
    
    cross_attn = cross_attention(
        first_layer,
        encoder_output,
        Wq_cross,
        Wk_cross,
        Wv_cross,
        Wo_cross,
        num_heads
    )
    
    second_residual = residual(first_layer, cross_attn)
    second_layer = layer_norm(second_residual, gamma2, beta2)
    
    ffn = feed_forward(second_layer, W1, b1, W2, b2)
    
    final_res = residual(second_layer, ffn)
    final_norm = layer_norm(final_res, gamma3, beta3)
    
    return final_norm

# Run code

In [2]:
import numpy as np

np.random.seed(42)

# -----------------------------
# Model dimensions
# -----------------------------
batch_size = 1
decoder_seq_len = 3
encoder_seq_len = 4

d_model = 4
num_heads = 2
d_ff = 8


# -----------------------------
# Decoder input
# -----------------------------
decoder_input = np.random.randn(
    batch_size,
    decoder_seq_len,
    d_model
)

print("Decoder input shape:", decoder_input.shape)


# -----------------------------
# Encoder output
# -----------------------------
encoder_output = np.random.randn(
    batch_size,
    encoder_seq_len,
    d_model
)

print("Encoder output shape:", encoder_output.shape)

Decoder input shape: (1, 3, 4)
Encoder output shape: (1, 4, 4)


In [3]:
# -----------------------------
# Self-attention weights
# -----------------------------
Wq_self = np.random.randn(d_model, d_model)
Wk_self = np.random.randn(d_model, d_model)
Wv_self = np.random.randn(d_model, d_model)
Wo_self = np.random.randn(d_model, d_model)


# -----------------------------
# Cross-attention weights
# -----------------------------
Wq_cross = np.random.randn(d_model, d_model)
Wk_cross = np.random.randn(d_model, d_model)
Wv_cross = np.random.randn(d_model, d_model)
Wo_cross = np.random.randn(d_model, d_model)


# -----------------------------
# FFN weights
# -----------------------------
W1 = np.random.randn(d_model, d_ff)
b1 = np.random.randn(d_ff)

W2 = np.random.randn(d_ff, d_model)
b2 = np.random.randn(d_model)

In [4]:
gamma1 = np.ones(d_model)
beta1 = np.zeros(d_model)

gamma2 = np.ones(d_model)
beta2 = np.zeros(d_model)

gamma3 = np.ones(d_model)
beta3 = np.zeros(d_model)

In [5]:
output = decoder_block(
    decoder_input,
    encoder_output,

    Wq_self,
    Wk_self,
    Wv_self,
    Wo_self,

    Wq_cross,
    Wk_cross,
    Wv_cross,
    Wo_cross,

    W1,
    b1,
    W2,
    b2,

    gamma1,
    beta1,
    gamma2,
    beta2,
    gamma3,
    beta3,

    num_heads
)

print("Decoder output shape:", output.shape)
print(output)

Decoder output shape: (1, 3, 4)
[[[-0.03314423  1.62394921 -1.01490129 -0.5759037 ]
  [-0.99166225  1.5854767  -0.70098744  0.10717299]
  [-1.16981491  1.53472844 -0.50615463  0.1412411 ]]]
